# Task 3 – Data Cleaning
## Customer Sales Dataset

**Objective:** Transform a deliberately messy dataset into a clean, analysis-ready dataset and document each decision.

### Dataset note
This notebook uses a reproducible, locally generated customer-sales dataset (500 base records) with intentionally introduced missing values, duplicates, inconsistent text/date formats, numeric values stored as text, and implausible numeric values. This makes the exercise self-contained and reproducible without depending on a changing download link. The dataset is synthetic and must not be represented as real customer information.

**Files:** `Task_3_Messy_Customer_Data.csv` (input) and `Task_3_Cleaned_Customer_Data.csv` (output). Keep both CSV files in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Load the messy dataset

In [ ]:
# If the CSV is not in the current folder, update this path.
df = pd.read_csv('Task_3_Messy_Customer_Data.csv')
print('Dataset shape:', df.shape)
display(df.head())
display(df.info())

## 3. Initial data quality report

In [ ]:
def quality_report(data):
    report = pd.DataFrame({
        'dtype': data.dtypes.astype(str),
        'null_count': data.isna().sum(),
        'null_percent': (data.isna().mean() * 100).round(2),
        'unique_values': data.nunique(dropna=True)
    })
    return report

print('Exact duplicate rows:', df.duplicated().sum())
display(quality_report(df))
display(df.describe(include='all').T)

### Initial quality observations
- Null counts identify columns that need a missing-value strategy.
- Exact duplicates are checked across all columns before removal.
- `Age`, `Total_Spend`, and `Order_Count` may contain invalid ranges or text-formatted values.
- Gender and City contain inconsistent capitalisation/spacing.
- Purchase dates contain multiple formats and need parsing.
- Some numeric fields may be inferred as `object` because text and numbers are mixed.

## 4. Missing data strategy and justification

| Column | Strategy | Reason |
|---|---|---|
| `Customer_ID` | Delete rows if missing | It is the record identifier; imputing an ID would invent identity. |
| `Purchase_Date` | Delete rows if unparseable/missing | A valid transaction date is essential for time-based analysis; inventing a date is unsafe. |
| `Gender` | Mode imputation | Small number of missing categorical values; mode is a simple, transparent choice. |
| `City` | Mode imputation | Missing city is categorical; mode keeps the row while using the most common category. |
| `Age` | Median imputation | Numeric data can be skewed and median is less affected by extreme ages. |
| `Total_Spend` | Median imputation | Spend is typically skewed; median is more robust than mean. |
| `Order_Count` | Median imputation | Count is numeric and median is robust to unusual values. |

**Important:** Invalid values are first converted to missing values. Imputation is performed only after parsing and range validation.

## 5. Remove exact duplicate rows

In [ ]:
before_rows = len(df)
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
removed_duplicates = before_rows - len(df)

print('Rows before:', before_rows)
print('Exact duplicates removed:', removed_duplicates)
print('Rows after:', len(df))

## 6. Standardise text and date formats

In [ ]:
# IDs remain strings so leading zeros or letter prefixes are preserved.
df['Customer_ID'] = df['Customer_ID'].astype('string').str.strip()
df['Name'] = df['Name'].astype('string').str.strip().str.title()

# Standardise gender labels.
gender = df['Gender'].astype('string').str.strip().str.lower()
gender_map = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female',
    'other': 'Other'
}
df['Gender'] = gender.map(gender_map)

# Trim whitespace and standardise city names.
df['City'] = df['City'].astype('string').str.strip().str.title()

# Parse mixed date formats. Invalid/unrecognised dates become NaT.
df['Purchase_Date'] = pd.to_datetime(
    df['Purchase_Date'], errors='coerce', format='mixed', dayfirst=True
)

display(df[['Gender', 'City', 'Purchase_Date']].head(10))
print('Unparsed/missing dates:', df['Purchase_Date'].isna().sum())

### Standardisation decision
Gender values such as `m`, `Male`, and `FEMALE` are mapped to consistent labels. City names are stripped of extra spaces and converted to title case. Dates are converted to pandas `datetime64`; unparseable values become `NaT` and are handled using the documented row-deletion rule.

## 7. Correct numeric data types

In [ ]:
# Remove currency symbols, commas, and spaces before numeric conversion.
spend_text = (df['Total_Spend'].astype('string')
              .str.replace(r'[₹,$\\s]', '', regex=True)
              .str.replace(',', '', regex=False))
df['Total_Spend'] = pd.to_numeric(spend_text, errors='coerce')

# Extract numeric age from values such as '34 yrs'.
age_text = df['Age'].astype('string').str.extract(r'(-?\\d+(?:\\.\\d+)?)', expand=False)
df['Age'] = pd.to_numeric(age_text, errors='coerce')

df['Order_Count'] = pd.to_numeric(df['Order_Count'], errors='coerce')

display(df.dtypes)
display(df[['Age', 'Total_Spend', 'Order_Count']].head(10))

## 8. Outlier and range-anomaly detection

In [ ]:
numeric_cols = ['Age', 'Total_Spend', 'Order_Count']

def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_rows = []
for col in numeric_cols:
    low, high = iqr_bounds(df[col].dropna())
    mask = (df[col] < low) | (df[col] > high)
    outlier_rows.append({
        'column': col, 'IQR_lower_bound': low, 'IQR_upper_bound': high,
        'IQR_outlier_count': int(mask.sum())
    })

display(pd.DataFrame(outlier_rows))

### Outlier decisions
IQR flags statistical extremes, but an extreme value is not automatically an error. This dataset also has known business-validity rules:
- Age must be between 18 and 100.
- Total spend must be between 0 and 200,000 for this exercise.
- Order count must be between 1 and 100.

Values outside these explicit ranges are treated as invalid and converted to missing, then imputed using the column median. Values that are only IQR outliers but remain inside the validity ranges are retained: unusually high spending can be genuine customer behaviour. These thresholds are exercise assumptions and should be confirmed with a real business owner before production use.

In [ ]:
# Apply documented domain/range checks (not IQR-only deletion).
df.loc[~df['Age'].between(18, 100), 'Age'] = np.nan
df.loc[(df['Total_Spend'] < 0) | (df['Total_Spend'] > 200000), 'Total_Spend'] = np.nan
df.loc[~df['Order_Count'].between(1, 100), 'Order_Count'] = np.nan

print('Invalid values converted to missing values; valid IQR extremes retained.')

## 9. Apply missing-value treatment

In [ ]:
# Remove rows where essential identity/date information is unavailable.
rows_before_essential_drop = len(df)
df = df.dropna(subset=['Customer_ID', 'Purchase_Date']).copy()
essential_rows_removed = rows_before_essential_drop - len(df)

# Categorical mode imputation.
for col in ['Gender', 'City']:
    mode_values = df[col].mode(dropna=True)
    if not mode_values.empty:
        df[col] = df[col].fillna(mode_values.iloc[0])

# Numeric median imputation.
for col in ['Age', 'Total_Spend', 'Order_Count']:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print('Rows removed for missing essential ID/date:', essential_rows_removed)
print('Remaining null counts:')
display(df.isna().sum())

## 10. Final data type correction and validation

In [ ]:
df['Customer_ID'] = df['Customer_ID'].astype('string')
df['Name'] = df['Name'].astype('string')
df['Gender'] = df['Gender'].astype('string')
df['City'] = df['City'].astype('string')
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Age'] = df['Age'].round().astype('Int64')
df['Total_Spend'] = df['Total_Spend'].astype('float64')
df['Order_Count'] = df['Order_Count'].round().astype('Int64')

# Check constraints and uniqueness.
assert df['Customer_ID'].notna().all()
assert df['Purchase_Date'].notna().all()
assert df['Age'].between(18, 100).all()
assert df['Total_Spend'].between(0, 200000).all()
assert df['Order_Count'].between(1, 100).all()

display(df.dtypes)
display(df.head())
print('Remaining null cells:', int(df.isna().sum().sum()))
print('Remaining duplicate rows:', int(df.duplicated().sum()))

## 11. Before vs. after summary

In [ ]:
# Capture original and final data-quality measures.
before_summary = {
    'Stage': 'Before cleaning',
    'Rows': len(pd.read_csv('Task_3_Messy_Customer_Data.csv')),
    'Null cells': int(pd.read_csv('Task_3_Messy_Customer_Data.csv').isna().sum().sum()),
    'Duplicate rows': int(pd.read_csv('Task_3_Messy_Customer_Data.csv').duplicated().sum()),
    'Dtype accuracy': 'Mixed / requires correction'
}
after_summary = {
    'Stage': 'After cleaning',
    'Rows': len(df),
    'Null cells': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Dtype accuracy': 'Corrected and validated'
}
comparison = pd.DataFrame([before_summary, after_summary]).set_index('Stage')
display(comparison)

print('Before dtypes:')
display(pd.read_csv('Task_3_Messy_Customer_Data.csv').dtypes.astype(str))
print('After dtypes:')
display(df.dtypes.astype(str))

## 12. Save cleaned dataset

In [ ]:
output_file = 'Task_3_Cleaned_Customer_Data.csv'
df.to_csv(output_file, index=False)
print(f'Cleaned dataset saved to: {output_file}')
print('Final shape:', df.shape)

## 13. Final observations and conclusion

1. Duplicate records were identified and removed to prevent double counting.
2. Inconsistent category labels, spaces, and date formats were standardised.
3. Numeric fields stored as text were converted to numeric types; malformed values became missing.
4. Implausible values were handled using explicit range rules. Valid statistical extremes were retained rather than automatically deleted.
5. Missing values were handled using column-specific strategies, and rows missing essential identifiers or dates were removed.
6. The final dataset was validated and exported to a separate CSV file.

**Conclusion:** The cleaning workflow produces a consistent, analysis-ready customer-sales dataset while documenting assumptions and preserving plausible customer behaviour. In real projects, confirm business rules and imputation choices with stakeholders before applying them to production data.

## 14. Reproducibility checklist

- [ ] Keep this notebook and the messy CSV in the same folder.
- [ ] Install dependencies if needed: `pip install pandas numpy matplotlib jupyter`
- [ ] Run all notebook cells from top to bottom.
- [ ] Confirm the before/after summary and validation assertions.
- [ ] Open `Task_3_Cleaned_Customer_Data.csv` and check the exported result.
- [ ] Submit the notebook, input CSV, and cleaned output CSV together.